# Movie Recommendation System

### - Project Objective

The goal of this project is to build a personalized movie recommendation system using collaborative filtering techniques on Databricks Free Edition.

The system predicts:

“Which movies is a user most likely to rate highly?”

This is achieved using the ALS (Alternating Least Squares) algorithm from Apache Spark MLlib.

## Full Pipeline Architecture

Bronze Layer → Raw CSV data

Silver Layer → Cleaned & structured data

ML Layer → ALS model training

Evaluation Layer → RMSE performance validation

Recommendation Layer → Manual generation of top-N movies

Presentation Layer → Join with movie titles

🧠 Technical Strengths of This Project

This project demonstrates:

Distributed data processing using Spark

Machine learning with ALS

Matrix factorization concepts

Performance evaluation using RMSE

Handling Unity Catalog limitations

Personalized ranking using window functions

End-to-end ML pipeline design

💼 Resume-Ready Description

You could describe this project as:

Built a scalable movie recommendation system using Apache Spark ALS on Databricks Community Edition. Implemented collaborative filtering with matrix factorization, evaluated model performance using RMSE, and generated personalized top-N recommendations by manually constructing user–item prediction pipelines compatible with Unity Catalog constraints.


###  Data Ingestion (Bronze Layer – Raw Data)

The project begins by loading two datasets:

Ratings dataset (userId, movieId, rating)

Movies dataset (movieId, title, genres)

These files come from the MovieLens dataset, which is widely used for recommendation system research.

At this stage:

Data is stored exactly as received

No transformation or cleaning is applied

Schema is inferred automatically

This layer represents raw, unprocessed data.


### Data Cleaning & Preparation (Silver Layer)

Before training the model, the data must be cleaned and standardized.

The cleaning process includes:

✅ Selecting Relevant Columns

Only essential columns are retained:

userId

movieId

rating

Unnecessary columns like timestamp are removed.

✅ Data Type Standardization

The algorithm requires:

userId → integer

movieId → integer

rating → float

Data type conversion ensures compatibility with the ML algorithm.

✅ Handling Missing Values

Any rows with null values are removed to prevent training errors.

After this stage:

Data is clean

Schema is consistent

Dataset is ready for machine learning

### Model Building – ALS Algorithm

The recommendation engine is built using ALS (Alternating Least Squares).

🔎 What is ALS?

ALS is a collaborative filtering algorithm that performs matrix factorization.

Conceptually:

We start with a user–movie rating matrix like:

	Movie A	Movie B	Movie C
User 1	5	?	3
User 2	?	4	?

Many entries are missing.

ALS:

Learns hidden user preferences (latent features)

Learns hidden movie characteristics

Predicts missing ratings

🎯 Key Model Configurations

userCol → identifies users

itemCol → identifies movies

ratingCol → actual rating value

coldStartStrategy = "drop"
Removes users or movies not seen during training

nonnegative = True
Ensures predicted ratings are not negative

🏋️ Training & Testing Split

The dataset is split into:

80% training data

20% testing data

This ensures:

Model learns patterns from training data

Performance is evaluated on unseen test data

### Model Evaluation

To measure model accuracy, we use:

RMSE (Root Mean Squared Error)

RMSE measures the average difference between:

Actual rating

Predicted rating

Lower RMSE means:

Predictions are closer to real user ratings

Model is more accurate

This ensures the recommendation system is statistically validated.

### Generating Recommendations (Manual Method)

Normally, Spark provides a built-in function to generate recommendations for all users.

However, in Databricks Free Edition with Unity Catalog:

Some higher-order Spark functions are restricted

Built-in recommendation methods are blocked

Therefore, recommendations are generated manually.

Step A: Identify All Users

First, extract the list of unique users.

This ensures recommendations are generated for every user in the dataset.

Step B: Identify All Movies

Extract all unique movies available in the dataset.

This defines the candidate pool for recommendations.

Step C: Create User–Movie Combinations

All possible user–movie pairs are generated.

For example:

If:

1,000 users

1,700 movies

Then:
1.7 million possible combinations are created.

This simulates asking:

“What rating would User X give to Movie Y?”

Step D: Predict Ratings for All Pairs

The trained ALS model predicts ratings for every user–movie combination.

Now we have:

Predicted score for each user

For every possible movie

Step E: Rank Movies per User

To personalize recommendations:

Movies are sorted by predicted rating

Ranking is done separately for each user

This ensures:
Each user gets different recommendations.

Step F: Select Top 5 Movies

For each user:

Only the top 5 highest predicted movies are selected.

These become the final recommendations.

### Final Output – Human Readable Recommendations

The predicted results initially contain only movie IDs.

To make recommendations meaningful:

Results are joined with the movie metadata table

Movie titles and genres are added

Final output example:

userId	Movie Title	Predicted Rating

Now the system produces personalized, understandable recommendations.



In [0]:
# Load ratings data
ratings_df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("/Volumes/workspace/default/movie_volume/ratings.csv")

# Load movies data
movies_df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("/Volumes/workspace/default/movie_volume/movies.csv")

ratings_df.show(5)
movies_df.show(5)


In [0]:
from pyspark.sql.functions import col

ratings_clean = ratings_df \
    .select(
        col("userId").cast("int"),
        col("movieId").cast("int"),
        col("rating").cast("float")
    ) \
    .dropna()

movies_clean = movies_df \
    .select(
        col("movieId").cast("int"),
        col("title"),
        col("genres")
    ) \
    .dropna()

ratings_clean.printSchema()
movies_clean.printSchema()

In [0]:
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator

# Split data
train, test = ratings_clean.randomSplit([0.8, 0.2])

# Define ALS model
als = ALS(
    userCol="userId",
    itemCol="movieId",
    ratingCol="rating",
    coldStartStrategy="drop",
    nonnegative=True
)

# Train
model = als.fit(train)

# Predict
predictions = model.transform(test)


In [0]:
evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="rating",
    predictionCol="prediction"
)

rmse = evaluator.evaluate(predictions)
print("RMSE:", rmse)


Generate Movie Recommendations

In [0]:
users = ratings_clean.select("userId").distinct()
movies_df = ratings_clean.select("movieId").distinct()

from pyspark.sql.functions import col

user_movie = users.crossJoin(movies_df)



In [0]:
predictions = model.transform(user_movie)


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window = Window.partitionBy("userId").orderBy(col("prediction").desc())

top_recommendations = predictions.withColumn(
    "rank",
    row_number().over(window)
).filter(col("rank") <= 5)

top_recommendations.show()


In [0]:
final = top_recommendations.join(movies_clean, "movieId")

final.select("userId", "title", "prediction").show(truncate=False)
